# lexicon

> Word and phrase rules: banned vocabulary, hedges, fillers, and splices

In [1]:
#| default_exp lexicon

In [2]:
#| hide
from nbdev.showdoc import *

The phrase-level rules: banned vocabulary with plain replacements, hedge phrases, noting fillers, filler transitions, splice punctuation, consequence glue, and markdown emphasis. Each is a regex over a block's scrubbed text, needs no linguistic analysis, and runs in microseconds.

## Prose linters, and why this one exists

Linting prose with word lists is an established practice. [Vale](https://vale.sh) is the mainstream tool: a fast command-line linter that runs YAML-defined style packages, used by the documentation teams at GitLab, Microsoft, and others. [write-good](https://github.com/btford/write-good) checks for passive voice, weasel words, and clichés in English prose. [proselint](https://github.com/amperser/proselint) collects usage advice from writers like Strunk, White, and Garner into checks, and its authors describe the approach in their paper "proselint: the linting of science prose, and the science of prose linting". The [GOV.UK style guide](https://www.gov.uk/guidance/style-guide/a-to-z-of-gov-uk-style#words-to-avoid) maintains the words-to-avoid list that GDS editors apply to a whole government's content, published under the Open Government Licence v3.

slopometer does not wrap any of them. It needs weighted findings bound to `write_docs` tell numbers, character spans that survive into file edits, and one deterministic score, and none of those tools produces that shape. Their curated vocabulary is still valuable: the banned list below merges the `write_docs` lists with entries adapted from the GOV.UK words-to-avoid list (OGL v3, attributed here). Vale's engine stays unused, and so do its packages, because every entry we adopt has to pass the same precision bar as our own.

In [3]:
#| export
from fastcore.utils import *
from slopometer.core import *
from slopometer.segment import *


In [4]:
from fastcore.test import *

## The lexicon factory

Four rules share one mechanism: scan for words or phrases from a vocabulary, and attach the plain replacement as the suggestion where one exists. `lex_rule` builds and registers such a rule. With `suffix=True` a key also matches its inflections, which lets one entry catch "streamline", "streamlines", and "streamlining". The fixture sentence from the segmentation notebook shows it at work:

In [5]:
#| export
def lex_rule(
    name, # Rule name, as in `core.rule`
    tell, # `write_docs` tell number
    weight, # Tier the findings carry
    lex, # Vocabulary: word or phrase -> plain replacement, or None where deletion is the fix
    suffix=False, # Also match inflected forms of each key?
):
    "Build and register a phrase rule that scans for `lex` entries case-insensitively"
    tail = r'\w*' if suffix else ''
    pat = re.compile(r'\b(' + '|'.join(re.escape(k) for k in sorted(lex, key=len, reverse=True)) + r')' + tail + r'\b', re.I)
    def _f(txt): return [Finding(name, tell, m.start(), m.end(), m.group(), weight, lex.get(m.group(1).lower())) for m in pat.finditer(txt)]
    _f.__name__,_f.__doc__ = f'find_{name}',f'Scan for the {name} vocabulary'
    return rule(name, tell=tell, weight=weight, level='phrase')(_f)

## Banned vocabulary

The entries merge three sources. The `write_docs` kill-on-sight list and its replacement table come first. Entries adapted from the [GOV.UK words-to-avoid list](https://www.gov.uk/guidance/style-guide/a-to-z-of-gov-uk-style#words-to-avoid) (contains public sector information licensed under the Open Government Licence v3.0) come second: "liaise", "overarching", "countless", "incentivize". Marketing cliches round it out. Every entry passed one filter: in technical reference prose the word is wrong in effectively every occurrence, whatever its part of speech. Words that are wrong only in one sense or one part of speech ("optimize" when nothing is optimized, "leverage" as a verb, "navigate" outside a UI) wait for the syntax notebook, where the parse can tell the senses apart. "leverage" appears here anyway: its noun sense is finance vocabulary that reference prose about software has no use for.

In [6]:
#| export
banned = {'utilize': 'use', 'utilise': 'use', 'leverage': 'use', 'facilitate': 'help', 'robust': 'strong',
    'comprehensive': 'complete', 'seamless': 'smooth', 'enhance': 'improve', 'streamline': None,
    'empower': None, 'foster': 'encourage', 'pivotal': None, 'a testament to': None, 'realm': None,
    'landscape': None, 'delve': None, 'myriad': None, 'plethora': None, 'paradigm': None,
    'synergy': None, 'holistic': None, 'catalyze': None, 'catalyse': None, 'juxtapose': None,
    'tapestry': None, 'embark': None, 'endeavor': None, 'endeavour': None, 'encompass': None,
    'multifaceted': None, 'elucidate': 'explain', 'nuanced': None, 'liaise': None, 'overarching': None,
    'countless': 'many', 'incentivize': None, 'incentivise': None, 'game-changer': None,
    'cutting-edge': None, 'best-in-class': None, 'deliver results': None,
    # adopted from the GOV.UK words-to-avoid list (OGL v3) and no-cliches (MIT); evidence below
    'initiate': 'start', 'tackle': 'solve', 'a clean slate': None, 'in a nutshell': None}

find_banned = lex_rule('banned', tell=None, weight=KILL, lex=banned, suffix=True)

In [7]:
find_banned(scrub('widget leverages a robust paradigm to deliver results.'))

[[10] banned: 'leverages' -> 'use',
 [10] banned: 'robust' -> 'strong',
 [10] banned: 'paradigm',
 [10] banned: 'deliver results']

## Hedges, noting fillers, and filler transitions

Three more vocabularies use the same factory. Hedging (tell 7) is restricted to phrases that are hedges in every context. The bare modals "may" and "might" stay out: "callers may pass `None`" grants permission, and telling permission from hedging needs the contract, which no word list holds. Noting fillers (tell 8) and filler transitions (tell 19) carry no such ambiguity, and their entries come straight from the `write_docs` tell text.

In [8]:
#| export
hedges = {'potentially': None, 'in some cases': None, 'should generally': None, 'may or may not': None,
    'one might argue': None, 'it could be argued': None, 'arguably': None, 'more or less': None,
    # weasel words adopted from write-good's weasel-words (MIT); evidence below
    'very': None, 'quite': None, 'extremely': None, 'fairly': None, 'exceedingly': None, 'remarkably': None,
    'surprisingly': None, 'interestingly': None, 'clearly': None, 'completely': None, 'largely': None,
    'mostly': None, 'relatively': None, 'usually': None, 'significantly': None, 'substantially': None,
    'huge': None, 'tiny': None, 'vast': None, 'few': None, 'several': None, 'excellent': None}
noting = {'note that': None, "it's worth noting": None, 'it is worth noting': None, 'worth mentioning': None,
    'importantly': None, 'notably': None, 'keep in mind': None, 'bear in mind': None, 'it should be noted': None}
transitions = {'furthermore': 'and', 'moreover': 'and', 'additionally': 'also', 'in conclusion': None,
    'when it comes to': None, 'in the realm of': None}

find_hedges = lex_rule('hedges', tell=7, weight=SMELL, lex=hedges)
find_noting = lex_rule('noting', tell=8, weight=SMELL, lex=noting)
find_transitions = lex_rule('transitions', tell=19, weight=SMELL, lex=transitions)

In [9]:
test_eq(find_hedges(scrub('The result may be `None` when the key is missing.')), [])
find_hedges('This could potentially improve performance in some cases.') + \
    find_noting("It's worth noting that the cache is per-process.") + \
    find_transitions('Furthermore, the config is optional.')

[[3] hedges (tell 7, hedging): 'potentially',
 [3] hedges (tell 7, hedging): 'in some cases',
 [3] noting (tell 8, noting fillers): "It's worth noting",
 [3] transitions (tell 19, filler transitions): 'Furthermore' -> 'and']

## Splice punctuation and consequence glue

Tell 1 bans clauses joined instead of separated. Three joins are detectable from characters alone, and each is wrong in reference prose every time: the em dash, its two-hyphen imitation, and the semicolon between words. A spaced single hyphen between word characters is the same habit typed lazily, and it cannot be a markdown bullet here because segmentation gives bullets their own blocks. The colon stays out of this rule: a colon can introduce a list or an example legitimately, and telling that from a clause splice takes the parse, not the characters. Consequence glue (tell 6) is the ", so" join, and `write_docs` treats every occurrence as wrong.

In [10]:
#| export
_splice = re.compile(r'—|(?<=\w) -- (?=\w)|(?<=\w); (?=\w)|(?<=\w) - (?=\w)')
_conseq = re.compile(r', so\b')

@rule('splice', tell=1, weight=KILL, level='phrase')
def find_splice(txt):
    "Em dashes, two-hyphen dashes, semicolon joins, and spaced-hyphen joins"
    return [Finding('splice', 1, m.start(), m.end(), m.group(), KILL) for m in _splice.finditer(txt)]

@rule('conseq', tell=6, weight=SMELL, level='phrase')
def find_conseq(txt):
    "The ', so' consequence join"
    return [Finding('conseq', 6, m.start(), m.end(), m.group(), SMELL) for m in _conseq.finditer(txt)]

In [11]:
test_eq(find_splice('Ranges like 3-5 and compound-words pass.'), [])
find_splice("It isn't just a poller - it's the liveness authority — or so it claims; nobody checked.") + \
    find_conseq('The cache is warm, so calls are fast.')

[[10] splice (tell 1, splices): ' - ',
 [10] splice (tell 1, splices): '—',
 [10] splice (tell 1, splices): '; ',
 [3] conseq (tell 6, consequence glue): ', so']

## Markdown emphasis

Position is the only emphasis mechanism in reference prose (tell 3). Bold and italic markers in running text are the markdown face of the intensifier habit, and this rule catches them where the syntax notebook's intensifier rule catches the adverbs.

In [12]:
#| export
_emph = re.compile(r'\*\*[^*\n]+\*\*|(?<![\w*])\*[^*\s][^*\n]*\*(?![\w*])|(?<![\w_])__[^_\n]+__(?![\w_])')

@rule('emphasis', tell=3, weight=SMELL, level='phrase')
def find_emphasis(txt):
    "Bold and italic markers in running prose"
    return [Finding('emphasis', 3, m.start(), m.end(), m.group(), SMELL) for m in _emph.finditer(txt)]

In [13]:
test_eq(find_emphasis('Multiplication a*b and glob patterns *.py pass.'), [])
find_emphasis("This **isn't going to cut it** and it's about *endurance*.")

[[3] emphasis (tell 3, emphasis devices): "**isn't going to cut it**",
 [3] emphasis (tell 3, emphasis devices): '*endurance*']

## Mined vocabularies

The prose-linting ecosystem holds decades of curated word lists, and the sources named at the top of this notebook supply candidates: write-good's weasel words, its `too-wordy` phrase list, its `no-cliches` list (all MIT), and the full GOV.UK words-to-avoid entry (OGL v3). Candidates do not enter the lexicons on provenance. Each one must fire on the meter's actual subjects, which are LLM-written responses, and must stay silent on the clean fixtures. The evidence run below counts each candidate across 5,493 recent assistant replies mined from local session transcripts, against theory4 and two clean READMEs as the false-positive check.

The verdicts teach the filter's value. Weasel words fire constantly on LLM output and never on the clean set, and 22 of 34 join the hedge lexicon. Eight wordy phrases fire and join with their mechanical replacements. The 723 cliches produce two adoptions, because LLMs do not write "a far cry" or "a tough row to hoe", and a list built for human writers mostly misses the target. Four GOV.UK entries fire and join the banned list; the rest describe failures LLM prose does not commit.

In [ ]:
#| eval: false
# The evidence run: fetch the source lists, count candidate hits on the reply corpus and the clean fixtures.
# Network and private transcripts keep this cell out of CI; rerun it when revisiting adoptions.
import httpx, re
from ghapi.skill import GhApi
def _extract(js): return [w for w in re.findall(r"'([^']+)'", js) if not w.startswith('./')]
lists = {name: _extract(httpx.get(f'https://raw.githubusercontent.com/{o}/{r}/HEAD/{p}').text)
    for name, o, r, p in [('weasel', 'btford', 'weasel-words', 'weasel.js'),
        ('wordy', 'duereg', 'too-wordy', 'wordPhrases.js'), ('cliche', 'duereg', 'no-cliches', 'cliches.js')]}
def hits(phrase, txt): return len(re.findall(r'\b' + re.escape(phrase.lower()) + r'\b', txt))
# corpus_txt: turn-final reply texts from the llmsurgery mirror; clean_txt: theory4 + clean READMEs
sorted(((w, hits(w, corpus_txt), hits(w, clean_txt)) for w in lists['weasel']), key=lambda r: -r[1])[:12]

In [14]:
#| export
wordy = {'all of': 'all', 'additional': 'more', 'a number of': 'some', 'along the lines of': 'like',
    'already existing': 'existing', 'adjacent to': 'next to', 'anticipate': 'expect',
    'in order to': 'to', 'going forward': 'from now on'}

find_wordy = lex_rule('wordy', tell=None, weight=SMELL, lex=wordy)

In [15]:
test_eq(find_hedges(scrub('`slopometer` scores each reply in about 30ms.')), [])
find_hedges('This is very fast and usually works.') + find_wordy('In order to run all of the tests, allow additional time.')

[[3] hedges (tell 7, hedging): 'very',
 [3] hedges (tell 7, hedging): 'usually',
 [3] wordy: 'In order to' -> 'to',
 [3] wordy: 'all of' -> 'all',
 [3] wordy: 'additional' -> 'more']

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()